# Semantic Recruitment Matcher - Full Pipeline
**Run order:** Cell 1 -> 2 -> 3 -> 4 -> 5 -> 6

In [8]:
# ============================================================
# CELL 1: Load HuggingFace datasets
# ============================================================ 
from datasets import load_dataset

jd_data = load_dataset(
    "lang-uk/recruitment-dataset-job-descriptions-english",
    split="train[:1000]"
)
cv_data = load_dataset(
    "lang-uk/recruitment-dataset-candidate-profiles-english",
    split="train[:1000]"
)

print(f"JD columns: {jd_data.column_names}")
print(f"CV columns: {cv_data.column_names}")
print(f"Loaded: {len(jd_data)} JDs, {len(cv_data)} CVs")

JD columns: ['Position', 'Long Description', 'Company Name', 'Exp Years', 'Primary Keyword', 'English Level', 'Published', 'Long Description_lang', 'id', '__index_level_0__']
CV columns: ['Position', 'Moreinfo', 'Looking For', 'Highlights', 'Primary Keyword', 'English Level', 'Experience Years', 'CV', 'CV_lang', 'id', '__index_level_0__']
Loaded: 1000 JDs, 1000 CVs


In [9]:
# ============================================================
# CELL 1.5: Check missing data percentage
# ============================================================
import pandas as pd

df_jd = jd_data.to_pandas()
df_cv = cv_data.to_pandas()

print("=== Tỷ lệ % Missing Data trong Job Descriptions (JD) ===")
jd_missing_percent = (df_jd.isna().sum() / len(df_jd)) * 100
print(jd_missing_percent.round(2).astype(str) + " %")

print("\n=== Tỷ lệ % Missing Data trong Candidate Profiles (CV) ===")
cv_missing_percent = (df_cv.isna().sum() / len(df_cv)) * 100
print(cv_missing_percent.round(2).astype(str) + " %")

=== Tỷ lệ % Missing Data trong Job Descriptions (JD) ===
Position                  0.0 %
Long Description          0.0 %
Company Name              0.0 %
Exp Years                 0.0 %
Primary Keyword           0.0 %
English Level            13.5 %
Published                 0.0 %
Long Description_lang     0.0 %
id                        0.0 %
__index_level_0__         0.0 %
dtype: object

=== Tỷ lệ % Missing Data trong Candidate Profiles (CV) ===
Position              0.0 %
Moreinfo              0.0 %
Looking For          61.2 %
Highlights           58.2 %
Primary Keyword       0.2 %
English Level         0.0 %
Experience Years      0.0 %
CV                    0.0 %
CV_lang               0.0 %
id                    0.0 %
__index_level_0__     0.0 %
dtype: object


In [10]:
# ============================================================
# CELL 2: Clean data and prepare embeddings
# ============================================================
import re

def clean_text(text):
    """Normalize text."""
    if not text or not isinstance(text, str):
        return ""
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def truncate_text(text, max_length=500):
    """Trim on word boundary."""
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    last_space = truncated.rfind(" ")
    return truncated[:last_space] if last_space > 0 else truncated

def truncate_to_bytes(text, max_bytes):
    """Trim by UTF-8 byte length."""
    encoded = text.encode('utf-8')
    if len(encoded) <= max_bytes:
        return text
    return encoded[:max_bytes].decode('utf-8', errors='ignore')

# --- Clean JD ---
cleaned_jd = []
skipped_jd = 0
for i, item in enumerate(jd_data):
    description = clean_text(item.get("Long Description"))
    if len(description) < 20:   # skip empty JD
        skipped_jd += 1
        continue
    position = clean_text(item.get("Position")) or "Unknown Position"
    keyword  = clean_text(item.get("Primary Keyword")) or ""
    company  = clean_text(item.get("Company Name")) or "Unknown Company"
    exp_years = clean_text(str(item.get("Exp Years") or ""))

    # Build JD embed text
    text_for_embed = truncate_text(f"{position}. {keyword}. {description}")

    cleaned_jd.append({
        "id":           i + 1,
        "position":     truncate_to_bytes(position, 300),
        "description":  truncate_to_bytes(description, 2000),
        "company":      truncate_to_bytes(company, 200),
        "keyword":      truncate_to_bytes(keyword, 300),
        "exp_years":    truncate_to_bytes(exp_years, 50),
        "text_for_embed": text_for_embed
    })

# --- Clean CV ---
cleaned_cv = []
skipped_cv = 0
for i, item in enumerate(cv_data):
    cv_text = clean_text(item.get("CV"))
    if len(cv_text) < 20:       # skip empty CV
        skipped_cv += 1
        continue
    position    = clean_text(item.get("Position")) or "Unknown Position"
    highlights  = clean_text(item.get("Highlights")) or ""
    keyword     = clean_text(item.get("Primary Keyword")) or ""
    exp_years   = clean_text(str(item.get("Experience Years") or ""))
    looking_for = clean_text(item.get("Looking For")) or ""

    # Build CV embed text
    text_for_embed = truncate_text(f"{position}. {highlights}. {cv_text}")

    cleaned_cv.append({
        "id":           i + 1,
        "position":     truncate_to_bytes(position, 300),
        "cv_text":      truncate_to_bytes(cv_text, 2000),
        "highlights":   truncate_to_bytes(highlights, 1000),
        "keyword":      truncate_to_bytes(keyword, 300),
        "exp_years":    truncate_to_bytes(exp_years, 50),
        "looking_for":  truncate_to_bytes(looking_for, 500),
        "text_for_embed": text_for_embed
    })

print(f"JD: kept {len(cleaned_jd)}, skipped {skipped_jd} (empty)")
print(f"CV: kept {len(cleaned_cv)}, skipped {skipped_cv} (empty)")

JD: kept 1000, skipped 0 (empty)
CV: kept 1000, skipped 0 (empty)


In [11]:
# ============================================================
# CELL 3: Load model and embed data
# ============================================================
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

print("Embedding JDs...")
jd_texts   = [item["text_for_embed"] for item in cleaned_jd]
jd_vectors = model.encode(jd_texts, show_progress_bar=True, batch_size=64)

print("\nEmbedding CVs...")
cv_texts   = [item["text_for_embed"] for item in cleaned_cv]
cv_vectors = model.encode(cv_texts, show_progress_bar=True, batch_size=64)

print(f"\nJD vectors shape: {jd_vectors.shape}")  # (N, 384)
print(f"CV vectors shape: {cv_vectors.shape}")   # (N, 384)

Embedding JDs...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

c:\Users\GIGABYTE\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)



Embedding CVs...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]


JD vectors shape: (1000, 384)
CV vectors shape: (1000, 384)


In [12]:
# ============================================================
# CELL 4: Recreate collections
# ============================================================
from pymilvus import (
    connections, utility, Collection,
    FieldSchema, CollectionSchema, DataType
)

connections.connect(host="localhost", port="19530")

# --- Drop old collections ---
for name in ["cvs", "job_descriptions"]:
    if utility.has_collection(name):
        Collection(name).drop()
        print(f"Dropped: {name}")

# --- Schema cho job_descriptions ---
jd_schema = CollectionSchema(
    fields=[
        FieldSchema(name="id",          dtype=DataType.INT64,        is_primary=True, auto_id=False),
        FieldSchema(name="vector",      dtype=DataType.FLOAT_VECTOR, dim=384),
        FieldSchema(name="position",    dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="description", dtype=DataType.VARCHAR,       max_length=2000),
        FieldSchema(name="company",     dtype=DataType.VARCHAR,       max_length=200),
        FieldSchema(name="keyword",     dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="exp_years",   dtype=DataType.VARCHAR,       max_length=50),
    ],
    description="Job descriptions",
    enable_dynamic_field=False
)

# --- Schema cho cvs ---
cv_schema = CollectionSchema(
    fields=[
        FieldSchema(name="id",          dtype=DataType.INT64,        is_primary=True, auto_id=False),
        FieldSchema(name="vector",      dtype=DataType.FLOAT_VECTOR, dim=384),
        FieldSchema(name="position",    dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="cv_text",     dtype=DataType.VARCHAR,       max_length=2000),
        FieldSchema(name="highlights",  dtype=DataType.VARCHAR,       max_length=1000),
        FieldSchema(name="keyword",     dtype=DataType.VARCHAR,       max_length=300),
        FieldSchema(name="exp_years",   dtype=DataType.VARCHAR,       max_length=50),
        FieldSchema(name="looking_for", dtype=DataType.VARCHAR,       max_length=500),
    ],
    description="Candidate CVs",
    enable_dynamic_field=False
)

jd_col = Collection(name="job_descriptions", schema=jd_schema)
cv_col = Collection(name="cvs",              schema=cv_schema)
print("Collections created.")

Dropped: cvs
Dropped: job_descriptions
Collections created.


In [13]:
# ============================================================
# CELL 5: Insert data into Milvus
# ============================================================

# --- Insert JDs ---
jd_insert = [
    {
        "id": int(item["id"]),
        "vector": vec.tolist(),
        "position": item["position"],
        "description": item["description"],
        "company": item["company"],
        "keyword": item["keyword"],
        "exp_years": item["exp_years"],
    }
    for item, vec in zip(cleaned_jd, jd_vectors)
]
jd_col.insert(jd_insert)
print(f"Inserted {len(cleaned_jd)} JDs")

# --- Insert CVs ---
cv_insert = [
    {
        "id": int(item["id"]),
        "vector": vec.tolist(),
        "position": item["position"],
        "cv_text": item["cv_text"],
        "highlights": item["highlights"],
        "keyword": item["keyword"],
        "exp_years": item["exp_years"],
        "looking_for": item["looking_for"],
    }
    for item, vec in zip(cleaned_cv, cv_vectors)
]
cv_col.insert(cv_insert)
print(f"Inserted {len(cleaned_cv)} CVs")

# --- Create index and load ---
index_params = {
    "index_type": "AUTOINDEX",
    "metric_type": "COSINE",
    "params": {}
}

jd_col.create_index(field_name="vector", index_params=index_params)
cv_col.create_index(field_name="vector", index_params=index_params)

jd_col.load()
cv_col.load()
print("Index and load done.")

Inserted 1000 JDs
Inserted 1000 CVs
Index and load done.


In [14]:
# ============================================================
# CELL 6: Test search
# ============================================================
# BUG FIX: encode one query at a time.
# Old code encoded the whole list as one string.
# That produced bad vectors and wrong results.
# Fix: encode inside the loop.

test_queries = [
    "Python backend developer with Django and REST API experience",
    "server-side engineer who builds web APIs and database integrations",
    "someone who thrives in fast-paced small teams with rapid iteration",
    "strong communicator who can lead cross-functional projects",
    "data person who understands both business side and technical implementation",
]

for query in test_queries:
    # Correct: encode one string -> shape (1, 384)
    query_vector = model.encode([query])

    results = cv_col.search(
        data=query_vector.tolist(),
        anns_field="vector",
        param={"metric_type": "COSINE"},
        limit=5,
        output_fields=["position", "keyword", "cv_text"]
    )

    print(f"\nQuery: '{query}'")
    print("=" * 60)
    for i, hit in enumerate(results[0]):
        print(f"  #{i+1} | Score: {hit.score:.4f} | {hit.entity.get('position')}")
        print(f"        Keyword: {hit.entity.get('keyword')}")
        print(f"        CV: {hit.entity.get('cv_text')[:80]}...")
    print("-" * 60)



Query: 'Python backend developer with Django and REST API experience'
  #1 | Score: 0.3680 | 1C developer
        Keyword: Flutter
        CV: 1 am an 1C developer. I deployed an 1C to typographical factory in Ukraine. Also...
  #2 | Score: 0.3618 | 1С Developer
        Keyword: Other
        CV: 1C 8.2, 8.3 programming, reports, processing, SKD. UPP, custom configurations, m...
  #3 | Score: 0.2884 | 1c Developer
        Keyword: Other
        CV: Worked on a mobile application for tracking trips...
  #4 | Score: 0.2795 | 3D Animator / 3D Artist
        Keyword: Unity
        CV: More than a two of experience in creating animation and working with UE. Worked ...
  #5 | Score: 0.2715 | 2D artist, graphic designer, ui designer
        Keyword: Design
        CV: I have 7 years experience as vector 2D artist working with microstock agencies M...
------------------------------------------------------------

Query: 'server-side engineer who builds web APIs and database integrations'
  #1 

In [15]:
# Run test queries and collect scores
test_queries = [
    "Python backend developer with Django and REST API experience",
    "server-side engineer who builds web APIs and database integrations",
    "someone who thrives in fast-paced small teams with rapid iteration",
    "strong communicator who can lead cross-functional projects",
    "data person who understands both business side and technical implementation",
]

all_scores = []
for q in test_queries:
    vec = model.encode([q])
    results = cv_col.search(
        data=vec.tolist(),
        anns_field="vector",
        param={"metric_type": "COSINE"},
        limit=10,
        output_fields=["position"]
    )
    for hit in results[0]:
        all_scores.append(hit.score)

all_scores.sort(reverse=True)
print(f"Max    : {max(all_scores):.4f}")
print(f"Min    : {min(all_scores):.4f}")
print(f"Mean   : {sum(all_scores)/len(all_scores):.4f}")
print(f"Top 10%: {all_scores[len(all_scores)//10]:.4f}")
print(f"Top 25%: {all_scores[len(all_scores)//4]:.4f}")

Max    : 0.5934
Min    : 0.2484
Mean   : 0.3978
Top 10%: 0.4696
Top 25%: 0.4490
